# Spam Detection

Classify messages as **spam** or **ham** with TF-IDF + logistic regression.

**Use case:** Email filters, chat moderation, phishing detection.

**Prerequisites:** `01-nlp-fundamentals.ipynb` introduces the concepts. This notebook uses `nlp_helpers.py` for preprocessing.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from nlp_helpers import download_nltk_data, preprocess_text

download_nltk_data()
print('Setup complete.')


## 1. Labeled messages

Small labeled set for demo—in production you would use thousands of emails with spam/ham labels.

In [ ]:
# Create a realistic spam/ham dataset (in production, use thousands of labeled emails)
messages = pd.DataFrame({
    'text': [
        "Meeting at 3pm in conference room B. Please confirm attendance.",
        "Your package has shipped. Track your order at link.com/track123.",
        "URGENT: Click here to claim your free prize! Limited time offer!!!",
        "Q4 report is ready for review. See attached.",
        "You've won $1,000,000! Reply with your bank details to claim.",
        "Team lunch on Friday. Let me know if you can make it.",
        "Act now! Guaranteed results. No credit card needed. Sign up today!",
        "Project deadline moved to next week. Updated schedule in Drive.",
        "Congratulations! You are selected for a special discount. Click now!",
        "Please review the attached invoice and let me know if you have questions.",
    ],
    'label': ['ham', 'ham', 'spam', 'ham', 'spam', 'ham', 'spam', 'ham', 'spam', 'ham']
})

messages['processed'] = messages['text'].apply(preprocess_text)
spam_vec = TfidfVectorizer(ngram_range=(1, 2))
X = spam_vec.fit_transform(messages['processed'])
y = (messages['label'] == 'spam').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

print("Spam detection accuracy:", round(accuracy_score(y_test, clf.predict(X_test)), 3))
print("\nTest on new message:")
test_msg = "Free gift! Click to claim your reward now!!!"
pred = clf.predict(spam_vec.transform([preprocess_text(test_msg)]))
print(f"  '{test_msg}'")
print(f"  Prediction: {'SPAM' if pred[0] else 'HAM'}")